# OW v3 matched final comparison
Attach the four completed core experiment outputs. Ablations use the same protocol but are compared separately on Validation during development.


In [ ]:
import json
from pathlib import Path
import pandas as pd

EXPECTED = ['baseline_effb7_ow_v3', 'multistream_no_seg_ow_v3', 'seg_no_multistream_ow_v3', 'full_veritas_ow_v3']
root = Path("/kaggle/input") if Path("/kaggle/input").exists() else Path.cwd()
frames = []
for path in root.rglob("experiment_results.csv"):
    df = pd.read_csv(path)
    if "experiment" not in df:
        continue
    df = df[df.experiment.isin(EXPECTED)]
    if not df.empty:
        frames.append(df.assign(source=str(path)))
if not frames:
    raise FileNotFoundError("Attach completed v3 full-training outputs")
results = pd.concat(frames, ignore_index=True)
required = {"protocol_hash", "split", "implementation_version", "epochs_completed", "n", "accuracy", "f1", "precision", "recall"}
if not required <= set(results.columns):
    raise ValueError("Incomplete result provenance")
if results.experiment.duplicated().any():
    raise ValueError("Attach exactly one final result per experiment")
if set(results.experiment) != set(EXPECTED):
    raise ValueError("All four final experiments, including B7 baseline, are required")
if not ((results.split == "test") & (results.implementation_version == "ow_v3") &
        (results.epochs_completed == 3) & (results.n == 10905)).all():
    raise ValueError("Only final full Test results are comparable; exclude smoke and partial runs")
if results.protocol_hash.nunique() != 1 or results.protocol_hash.isna().any():
    raise ValueError("Protocol/seed/mask sources differ; rerun a matched comparison")
scores = results[["accuracy", "f1", "precision", "recall"]]
if not ((scores >= 0) & (scores <= 1)).all().all():
    raise ValueError("Nonfinite or invalid classification metrics")
baseline = float(results.loc[results.experiment == EXPECTED[0], "accuracy"].iloc[0])
results["accuracy_delta_pp_vs_b7"] = 100 * (results.accuracy - baseline)
out = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd() / "kaggle_working"
out.mkdir(parents=True, exist_ok=True)
results.to_csv(out / "ow_v3_compare.csv", index=False)
print(results[["experiment", "accuracy", "accuracy_delta_pp_vs_b7", "f1", "precision", "recall"]].sort_values("accuracy", ascending=False).to_string(index=False))
print("Positive delta means higher accuracy in this run; repeat matched seeds before a superiority claim.")
print("Fake Dice measures teacher agreement; Real positive-mask area is separate.")
